# L16a: The Curse of Dimensionality and the Rise of Deep Q-Learning
This lecture revisits Q-learning and shows how it can be extended to problems with large or continuous state and action spaces. Tabular Q-learning stores one entry per state-action pair, which is infeasible once the state space is high-dimensional. _Deep Q-learning_ (DQN) replaces the Q-table with a neural network that maps a state to a vector of Q-values, one per action, and trains that network from past experience using two stabilization tricks: a replay buffer and a delayed target network.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __State why tabular Q-learning fails in high dimensions:__ Explain how the size of the state-action table grows with the number of features, and why this makes tabular updates infeasible for image-like or continuous inputs. Identify what a function approximator buys you in this regime.
> * __Write down the DQN learning rule:__ Identify the role of the main Q-network, the target network, and the replay buffer in the DQN update. State the form of the target value used in the mean squared loss, and explain why each of the three components is needed for stable training.
> * __Describe the practical details that make DQN work:__ Explain the warm-up phase, the fixed-size circular replay buffer, the periodic copy from main to target network, and the choice to take a single gradient step per mini-batch. State what each detail prevents.

The sources for this lecture include:

* __Stanford University__ – CS234: Reinforcement Learning: This course comprehensively introduces reinforcement learning, covering foundational algorithms and deep reinforcement learning methods, including Deep Q-Learning (DQN). [course link](https://web.stanford.edu/class/cs234/?utm_source=chatgpt.com) and [notes link](https://web.stanford.edu/class/cs234/modules.html)
* __University of Toronto__ – CSC311: Introduction to Machine Learning (Fall 2020): This undergraduate course introduces core machine learning concepts, including supervised learning, unsupervised learning, and reinforcement learning. Lecture 11 covers reinforcement learning basics, with foundational material relevant to Deep Q-Learning (DQN). [Course link](https://www.cs.toronto.edu/~rgrosse/courses/csc311_f20/) and [Lecture 11 notes link](https://www.cs.toronto.edu/~rgrosse/courses/csc311_f20/slides/lec11.pdf)
* __Mnih et al. (2015)__ – "Human-level control through deep reinforcement learning," _Nature_ 518, 529-533. The original DQN paper. [link](https://www.nature.com/articles/nature14236)

Let's go!
___

## Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We have explored different approaches to this problem:

* __Multiplicative weights__ approaches the probability of selecting an action based on past performance, but they do so in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight—even in changing environments, i.e., it minimizes regret.
* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Q-learning__ is a value-based method that estimates the long-term value (utility, satisfaction, happiness, etc) of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.

These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

<div>
    <center>
        <img src="figs/Fig-Q-Schematic.svg" width="500"/>
    </center>
</div>

## Review: Q-Learning Theory
Q-learning estimates the action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{\underbrace{Q_{t}(s,a)}_{\text{old value}}}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new value}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$


### Algorithm
__Initialize__ $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha_{t}$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$,the maximum number of iterations $\texttt{maxiter}$, and the convergence tolerance $\delta$. Set the $\texttt{converged}\gets\texttt{false}$. 

For $s\in\mathcal{S}$
1. Set the trial counter $t\gets{1}$
2. While $\texttt{converged} $ is $\texttt{false}$ __do__:
    1. Roll a random number $p\in[0,1]$. Compute $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$ where $K=|\mathcal{A}|$ is the number of actions.
    2. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$.
    3. Take action $a_{t}$, observe the reward $r$ from the __world__ and transition to the next state $s^{\prime}$.
    4. Update the state-action-value function: $Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a)\right)}_{\text{new information}}$.
    5. Update the state $s\leftarrow{s^{\prime}}$, the learning rate $\alpha_{t+1}\leftarrow\alpha_{t}$ and the counter $t\leftarrow{t+1}$
    6. Convergence check: If the $Q(s,a)$ has bounded change $\lVert{Q_{t+1}(s,a) - Q_{t}(s,a)}\rVert\leq\delta$, then the algorithm has converged. Set $\texttt{converged}\gets\texttt{true}$.
    7. Otherwise: if $t\geq\texttt{maxiter}$, then set $\texttt{converged}\gets\texttt{true}$ and notify the caller that the maximum iteration limit was reached without convergence. Proceed to next state.
    8. Otherwise: continue to the next iteration.
3. End While
4. End For

### Convergence
Q-learning converges to the optimal policy under two key theoretical conditions (assuming the Markov property holds for the environment):
* __Learning rate decay__: The learning rate $\alpha_{t}$ must satisfy $\sum_{t=0}^\infty \alpha_t(s, a) = \infty$ and $\sum_{t=0}^\infty \alpha_t^2(s, a) < \infty$ for all state-action pairs, ensuring sufficient initial updates while stabilizing over time. Setting $\alpha_{t+1} \gets \beta\alpha_{t}$ where $\beta<1$ is a common choice.
* __Infinite exploration__: All state-action pairs must be visited infinitely often. This condition holds for $\epsilon$-greedy policies with persistent exploration, i.e., $\epsilon_{t} > 0\,\,\forall{t}$.

See the [Q-learning lecture notes from L8c for more details](https://htmlview.glitch.me/?https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-8/L8c/CHEME-5820-L8c-QLearning-S2025.html).

___

## Deep Q-Learning (DQN)
Deep Q-learning is a variant of Q-learning that uses a deep neural network to approximate the Q-value function.

<img src="figs/Q-Learning-vs-Deep-Q-Learning.ppm.png" alt="Q-Learning vs Deep Q-Learning" width="500"/>

This approach allows for the handling of high-dimensional state spaces, such as images or continuous states, where traditional Q-learning would be infeasible due to the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality).
* _How does the approach work_? In this approach, the Q-value function $Q(s, a)$ is represented as a neural network, which takes the state $s$ as input and outputs the Q-values for all possible actions. The neural network is trained using the same Q-learning update rule, but with mini-batches of experiences sampled from a replay buffer to stabilize training.
* _Games_? This approach was made famous by [the DeepMind team in 2015](https://www.nature.com/articles/nature14236), where they used DQN to play Atari games directly from pixels. This approach achieved human-level performance on other games. For example, DQN was used as part of the policy network [pre-training phase for AlphaGo](https://doi.org/10.1038/nature16961), the first AI to defeat a professional human Go player, marking a milestone in AI. 
* _Other applications?_ DQN has been used in other applications such as operations management, e.g., a [DQN-based system was deployed in Google's data centers to optimize cooling, achieving a reported 30% reduction in energy consumption for cooling systems](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/) or [traffic signal control in smart cities](https://dl.acm.org/doi/10.1145/3219819.3220096), where DQN was used to optimize traffic light timings in real-time, leading to reduced congestion and improved traffic flow.

### Theory of DQN
A deep Q-learning agent learns a policy $\pi$ that maximizes the expected cumulative reward $R_t$ over time. Suppose the agent is tasked with making decisions over $T\rightarrow\infty$ steps.

For each episode, we sample for $t = 1,2,\ldots,T$: 

1. __Interaction with the environment__: At each time step $t$, the agent observes the current state $s_t$, selects an action $a_t$ (typically using an $\epsilon$-greedy policy based on the _Q-network_), and receives a reward $r_t$ and the next state $s_{t+1}$ from the environment.
2. __Experience replay__: Each transition tuple $(s_t, a_t, r_t, s_{t+1})$ is stored in a **replay buffer** (a finite-sized memory that we'll use for training). Instead of training on consecutive samples, the agent **samples random mini-batches** from this buffer. 
3. __Main Q-Network (function approximator)__: The core of DQN is a deep neural network $Q_{\theta}(s)\in\mathbb{R}^{|\mathcal{A}|}$ with (trainable) parameters $\theta$, which learns to approximate the optimal action-value function. The network takes a state as input and outputs a vector of Q-values, one per action; we write $[Q_{\theta}(s)]_{a}$ for the entry corresponding to action $a$.
4. __Target Q-Network__: To stabilize training, DQN uses a **target network** $Q^{\prime}_{\theta^{-}}(s)\in\mathbb{R}^{|\mathcal{A}|}$, which is a delayed copy of the main Q-network. The target network's parameters $\theta^-$ are updated periodically (e.g., every $C$ steps) by copying the weights from the main Q-network.


#### DQN Algorithm

__Initialize__ the parameters of the main Q-network $Q_{\theta}(s)$ to random values, and copy them to the target Q-network so that $\theta^{-}\leftarrow\theta$. Initialize an empty replay buffer $\mathcal{B}$ with maximum size $M$. Set the hyperparameters: the learning rate $\alpha>0$, the discount factor $\gamma\in(0,1)$, the exploration rate $\epsilon_{t}\in(0,1]$, the warm-up threshold $N_{\text{warm}}$ (minimum buffer size before training begins), the mini-batch size $B$, and the target-network sync interval $C$.
- For each episode, initialize the state to $s_{0}$ and:
   - For each time step $t=1,\ldots,T$:
        1. Roll a random number $p\in[0,1]$. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \arg\max_{a\in\mathcal{A}}\,[Q_{\theta}(s_{t})]_{a}$.
        2. Execute action $a_{t}$, observe the reward $r_{t}$ and next state $s_{t+1}$ from the _world_, and observe a done flag $d_{t}\in\{0,1\}$ that is $1$ if $s_{t+1}$ is terminal and $0$ otherwise.
        3. Store the transition $e_{t}=(s_{t}, a_{t}, r_{t}, s_{t+1}, d_{t})$ in the replay buffer: $\mathcal{B}\leftarrow\mathcal{B}\cup\{e_{t}\}$, evicting the oldest transition if $|\mathcal{B}|$ exceeds $M$.
        4. If $|\mathcal{B}|\geq N_{\text{warm}}$, sample a mini-batch of $B$ transitions $\{(s_{i}, a_{i}, r_{i}, s_{i+1}, d_{i})\}_{i=1}^{B}$ uniformly at random from $\mathcal{B}$.
        5. Compute the _target Q-value_ for each transition in the mini-batch using the _target Q-network_: $y_{i} = r_{i} + \gamma\,(1 - d_{i})\,\max_{a^{\prime}\in\mathcal{A}}\,[Q^{\prime}_{\theta^{-}}(s_{i+1})]_{a^{\prime}}$ for $i=1,2,\ldots,B$. The factor $(1 - d_{i})$ zeroes out the bootstrap term on terminal transitions.
        6. Compute the _mean squared loss_ over the $B$ experiences in the mini-batch using the action that was actually taken: $L(\theta) = \frac{1}{B}\sum_{i=1}^{B}\left(y_{i} - [Q_{\theta}(s_{i})]_{a_{i}}\right)^{2}$.
        7. Perform a _single_ gradient descent step to minimize the loss function $L(\theta)$ with respect to the parameters $\theta$ of the main Q-network: $\theta \leftarrow \theta - \alpha\,\nabla_{\theta}L(\theta)$.
            - _Why only a single step_? Each mini-batch is just a small sample of the environment's dynamics. The goal of DQN is _online learning_: the network parameters are continuously updated as new experiences come in. Forcing training to converge on each mini-batch risks _overfitting to that mini-batch_.
        8. Update the state $s_{t} \leftarrow s_{t+1}$.
        9. Every $C$ steps, update the target Q-network parameters: $\theta^{-} \leftarrow \theta$.
    - End For
- End For

## Practical details
Several practical details are important to DQN's success, primarily focused around the training process and the management of the replay buffer. These details are critical for DQN's stability and performance, especially in complex environments.

### Replay buffer management
The replay buffer is a key component of DQN, allowing the agent to learn from past experiences. The buffer stores transitions (state, action, reward, next state) and samples mini-batches for training. Thus, one obvious question is: _How do we manage the replay buffer_?

* In DQN, the **replay buffer has a fixed maximum size** (often denoted as $M$, e.g., 100,000 or 1,000,000, etc). This is done for memory efficiency and to ensure the agent focuses on more recent, relevant experiences.
* When the buffer reaches its maximum size, the oldest experiences are discarded to make room for new ones, e.g., in a first-in, first-out manner. This ensures that the agent learns from a diverse set of experiences and avoids overfitting to outdated information.
* This approach ensures that the replay buffer contains a mix of old and new experiences, prioritizes more recent experiences over time, and stays within a fixed memory footprint.

### Mini-batch construction
Each mini-batch is sampled randomly from the current contents of the buffer. Older experiences are still used for training as long as they remain in the buffer. However, once an experience is overwritten (evicted), it no longer contributes to future mini-batches. Thus, the mini-batch is a random sample of old and new experiences from the replay buffer. 

* __Vanilla DQN__: The basic replay buffer in DQN is typically implemented using a [circular buffer (ring buffer)](https://en.wikipedia.org/wiki/Circular_buffer#:~:text=In%20computer%20science%2C%20a%20circular,easily%20to%20buffering%20data%20streams.), which is conceptually similar to a fixed-size queue. This data structure has constant-time insert and sample operations, making storing and sampling experiences efficient.
* __Prioritized Replay DQN__: When using **prioritized experience replay** (where experiences are sampled with different probabilities based on their _importance_), a more advanced data structure is needed, e.g., a [binary heap](https://en.wikipedia.org/wiki/Binary_heap#:~:text=A%20binary%20heap%20is%20a,data%20structure%20for%20implementing%20heapsort.) or a [sum tree](https://en.wikipedia.org/wiki/Fenwick_tree). This allows for efficient sampling of experiences based on their priority.

### Training frequency
In practice, DQN does not train the network immediately at every step. Instead, training begins after the replay buffer has accumulated a minimum number of transitions (often called the warm-up period). This ensures that each mini-batch used for training contains diverse and meaningful experiences, stabilizing learning.

* __Warm-Up Phase__: The agent interacts with the environment and **stores transitions** in the replay buffer **without updating the Q-network** until the buffer reaches a threshold size (e.g., 1,000 or 10,000 transitions, depending on the problem).
* __Training Phase__: Training begins once the replay buffer exceeds the threshold. After that point, the network can be updated at every environment step. Alternatively, it can be updated at a fixed interval (e.g., every four environment steps) to save computation.
* Thus, a typical pattern is: (i) Fill the replay buffer to a minimum size, (ii) Start training, and (iii) Continue to fill the buffer with new experiences while training.

## Lab
In `L16c`, we will implement a simple Deep Q-Learning (DQN) agent. The goal is to train the agent to play a continuous-valued game that is not accessible to traditional Q-learning.

## Summary
Deep Q-learning replaces the tabular Q-function of standard Q-learning with a neural network that maps a state to a vector of Q-values, one per action. Training reuses the Q-learning target $r + \gamma\max_{a'}Q'(s', a')$ but stabilizes it with two additions: a replay buffer that decorrelates consecutive transitions and a delayed target network that holds the bootstrap target steady between periodic syncs. The same loop scales from low-dimensional toy problems to image-based control without changing the algorithm.

> __Key Takeaways:__
>
> * __DQN trades a table for a function approximator:__ The Q-table grows with the number of state-action pairs and becomes infeasible in high dimensions. A neural network shares parameters across states, generalizes to states it has never visited, and reduces the storage cost from one entry per state-action pair to one fixed-size weight vector.
> * __Replay buffer plus target network are the two stabilizers:__ Sampling random mini-batches from a fixed-size circular buffer breaks the temporal correlation between consecutive transitions, and the periodically synced target network keeps the bootstrap target from chasing the parameters being trained. Both are needed for stable learning.
> * __The loss is on the action that was actually taken:__ The mean squared loss compares the target value to $[Q_{\theta}(s_{i})]_{a_{i}}$, the Q-value of the action stored in the replay tuple, not to the whole vector. A single gradient step per mini-batch keeps training online and avoids overfitting to any one batch.

The companion lab notebook [L16c](../L16c/) implements a DQN agent on a continuous-valued control task and exercises every component covered here: replay buffer, target network, $\epsilon$-greedy exploration, and the mean squared bootstrap loss.
___